<a href="https://colab.research.google.com/github/SanviBallal/Factory-Shipping-Optimization/blob/main/07_model_development_and_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/feature_engineered_dataset.csv")

print("DATASET LOADED SUCCESSFULLY")
print("-" * 45)

print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset preview:")
display(df.head())

print("\nData types:")
print(df.dtypes)


DATASET LOADED SUCCESSFULLY
---------------------------------------------
Dataset shape: (10194, 39)

Column names:
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Country/Region', 'City', 'State/Province', 'Postal Code', 'Division', 'Region', 'Product ID', 'Product Name', 'Sales', 'Units', 'Gross Profit', 'Cost', 'Latitude', 'Longitude', 'Accuracy', 'Coordinate Source', 'Customer Location ID', 'Current Factory', 'Nearest Factory', 'Nearest Factory Distance (km)', 'Current Factory Distance (km)', 'Potential Distance Reduction (km)', 'Potential Distance Reduction (%)', 'Current Factory Is Nearest', 'Profit Margin (%)', 'Sales per Unit', 'Cost per Unit', 'Profit per Unit', 'Order Year', 'Order Month', 'Order Month Name', 'Order Quarter', 'Order Day of Week']

Dataset preview:


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,...,Current Factory Is Nearest,Profit Margin (%),Sales per Unit,Cost per Unit,Profit per Unit,Order Year,Order Month,Order Month Name,Order Quarter,Order Day of Week
0,1,US-2021-103800-CHO-MIL-31000,2024-01-03,30-06-2026,Standard Class,103800,United States,Houston,Texas,77095,...,False,64.923077,3.25,1.14,2.11,2024,1,January,1,Wednesday
1,2,US-2021-112326-CHO-TRI-54000,2024-01-04,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,...,False,65.333333,3.75,1.30,2.45,2024,1,January,1,Thursday
2,3,US-2021-112326-CHO-NUT-13000,2024-01-04,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,...,False,71.346705,3.49,1.00,2.49,2024,1,January,1,Thursday
3,4,US-2021-112326-CHO-SCR-58000,2024-01-04,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,...,False,69.444444,3.60,1.10,2.50,2024,1,January,1,Thursday
4,5,US-2021-141817-CHO-TRI-54000,2024-01-05,05-07-2026,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,...,True,65.333333,3.75,1.30,2.45,2024,1,January,1,Friday



Data types:
Row ID                                 int64
Order ID                              object
Order Date                            object
Ship Date                             object
Ship Mode                             object
Customer ID                            int64
Country/Region                        object
City                                  object
State/Province                        object
Postal Code                           object
Division                              object
Region                                object
Product ID                            object
Product Name                          object
Sales                                float64
Units                                  int64
Gross Profit                         float64
Cost                                 float64
Latitude                             float64
Longitude                            float64
Accuracy                             float64
Coordinate Source                     obje

In [2]:

# DAY 7 - MODELLING SUITABILITY ASSESSMENT

print("MODELLING SUITABILITY REPORT")
print("=" * 55)

print("\n1. FACTORY INFORMATION")
print("Number of factories:",
      df["Current Factory"].nunique())

print("Factories:")
print(sorted(df["Current Factory"].unique()))

print("\n2. PRODUCT INFORMATION")
print("Number of products:",
      df["Product Name"].nunique())

print("\n3. FINANCIAL INFORMATION")
print("Sales available:", df["Sales"].notna().all())
print("Gross Profit available:", df["Gross Profit"].notna().all())
print("Cost available:", df["Cost"].notna().all())

print("\n4. DISTANCE INFORMATION")
print("Current factory distance available:",
      df["Current Factory Distance (km)"].notna().all())

print("Nearest factory distance available:",
      df["Nearest Factory Distance (km)"].notna().all())

print("\n5. POTENTIAL ALLOCATION OPPORTUNITY")
print("Transactions with a closer alternative factory:",
      (~df["Current Factory Is Nearest"]).sum())

print("\n6. IMPORTANT DATA LIMITATION")
print("Verified shipping cost target: Not available")
print("Reliable delivery time target: Not available")

print("\nAssessment completed.")


MODELLING SUITABILITY REPORT

1. FACTORY INFORMATION
Number of factories: 5
Factories:
["Lot's O' Nuts", 'Secret Factory', 'Sugar Shack', 'The Other Factory', "Wicked Choccy's"]

2. PRODUCT INFORMATION
Number of products: 15

3. FINANCIAL INFORMATION
Sales available: True
Gross Profit available: True
Cost available: True

4. DISTANCE INFORMATION
Current factory distance available: True
Nearest factory distance available: True

5. POTENTIAL ALLOCATION OPPORTUNITY
Transactions with a closer alternative factory: 6967

6. IMPORTANT DATA LIMITATION
Verified shipping cost target: Not available
Reliable delivery time target: Not available

Assessment completed.


In [3]:

# DAY 7 - PRODUCT-WISE REALLOCATION OPPORTUNITY ANALYSIS

product_opportunities = (
    df.groupby(
        ["Product Name", "Current Factory"]
    )
    .agg(
        Transactions=("Row ID", "count"),
        Average_Current_Distance=(
            "Current Factory Distance (km)", "mean"
        ),
        Average_Nearest_Distance=(
            "Nearest Factory Distance (km)", "mean"
        ),
        Average_Potential_Reduction=(
            "Potential Distance Reduction (km)", "mean"
        ),
        Total_Potential_Reduction=(
            "Potential Distance Reduction (km)", "sum"
        ),
        Total_Sales=("Sales", "sum"),
        Total_Gross_Profit=("Gross Profit", "sum")
    )
    .reset_index()
)

product_opportunities = product_opportunities.sort_values(
    by="Average_Potential_Reduction",
    ascending=False
)

print("PRODUCT-WISE REALLOCATION OPPORTUNITIES")
print("=" * 65)

display(product_opportunities.round(2))

print("\nTOP 5 PRODUCTS BY AVERAGE POTENTIAL DISTANCE REDUCTION")
display(product_opportunities.head(5).round(2))


PRODUCT-WISE REALLOCATION OPPORTUNITIES


,Product Name,Current Factory,Transactions,Average_Current_Distance,Average_Nearest_Distance,Average_Potential_Reduction,Total_Potential_Reduction,Total_Sales,Total_Gross_Profit
9,Wonka Bar - Fudge Mallows,Lot's O' Nuts,1818,2165.80,826.33,1339.47,2435161.16,24890.40,16593.60
11,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts,1810,2125.55,802.98,1322.57,2393850.00,23574.95,16819.95
13,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts,2064,2142.28,833.31,1308.97,2701708.29,27874.80,19357.50
3,Hair Toffee,The Other Factory,4,1751.22,549.21,1202.01,4808.04,76.50,59.50
1,Fizzy Lifting Drinks,Sugar Shack,6,1728.16,539.26,1188.90,7133.38,78.75,47.25
12,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,2015,1885.17,804.86,1080.31,2176825.50,28485.00,18610.20
10,Wonka Bar - Milk Chocolate,Wicked Choccy's,2137,1843.70,820.89,1022.81,2185754.12,26867.75,17443.37
5,Laffy Taffy,Sugar Shack,10,1901.15,963.82,937.34,9373.36,53.73,33.48
2,Fun Dip,Sugar Shack,3,1538.03,610.91,927.12,2781.36,12.00,4.80
8,SweeTARTS,Sugar Shack,10,1759.22,897.69,861.52,8615.24,61.50,28.70



TOP 5 PRODUCTS BY AVERAGE POTENTIAL DISTANCE REDUCTION


,Product Name,Current Factory,Transactions,Average_Current_Distance,Average_Nearest_Distance,Average_Potential_Reduction,Total_Potential_Reduction,Total_Sales,Total_Gross_Profit
9,Wonka Bar - Fudge Mallows,Lot's O' Nuts,1818,2165.80,826.33,1339.47,2435161.16,24890.40,16593.60
11,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts,1810,2125.55,802.98,1322.57,2393850.00,23574.95,16819.95
13,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts,2064,2142.28,833.31,1308.97,2701708.29,27874.80,19357.50
3,Hair Toffee,The Other Factory,4,1751.22,549.21,1202.01,4808.04,76.50,59.50
1,Fizzy Lifting Drinks,Sugar Shack,6,1728.16,539.26,1188.90,7133.38,78.75,47.25


In [4]:

# DAY 7 - IDENTIFY NEAREST ALTERNATIVE FACTORY

alternative_summary = (
    df[df["Current Factory"] != df["Nearest Factory"]]
    .groupby(
        ["Product Name", "Current Factory", "Nearest Factory"]
    )
    .agg(
        Transactions=("Row ID", "count"),
        Average_Current_Distance=(
            "Current Factory Distance (km)", "mean"
        ),
        Average_Alternative_Distance=(
            "Nearest Factory Distance (km)", "mean"
        ),
        Average_Distance_Reduction=(
            "Potential Distance Reduction (km)", "mean"
        ),
        Total_Distance_Reduction=(
            "Potential Distance Reduction (km)", "sum"
        )
    )
    .reset_index()
)

alternative_summary = alternative_summary.sort_values(
    by="Transactions",
    ascending=False
)

print("NEAREST ALTERNATIVE FACTORY ANALYSIS")
print("=" * 65)

display(alternative_summary.round(2))

print("\nMOST COMMON ALTERNATIVE FACTORY PAIRS")

display(
    alternative_summary.groupby(
        ["Current Factory", "Nearest Factory"]
    )["Transactions"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)


NEAREST ALTERNATIVE FACTORY ANALYSIS


,Product Name,Current Factory,Nearest Factory,Transactions,Average_Current_Distance,Average_Alternative_Distance,Average_Distance_Reduction,Total_Distance_Reduction
45,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts,Wicked Choccy's,707,3287.82,953.19,2334.63,1650583.90
38,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,Lot's O' Nuts,667,3458.36,934.00,2524.35,1683744.04
30,Wonka Bar - Milk Chocolate,Wicked Choccy's,Lot's O' Nuts,663,3439.74,936.75,2503.00,1659487.62
29,Wonka Bar - Fudge Mallows,Lot's O' Nuts,Wicked Choccy's,620,3274.11,937.46,2336.66,1448728.15
37,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts,Wicked Choccy's,585,3247.92,912.93,2334.99,1365969.41
31,Wonka Bar - Milk Chocolate,Wicked Choccy's,Secret Factory,403,1216.21,567.32,648.89,261501.83
39,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,Secret Factory,371,1197.76,511.81,685.95,254487.05
34,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts,Secret Factory,347,2504.05,503.27,2000.79,694273.35
42,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts,Secret Factory,346,2546.85,543.99,2002.86,692988.90
26,Wonka Bar - Fudge Mallows,Lot's O' Nuts,Secret Factory,327,2566.43,547.22,2019.21,660282.15



MOST COMMON ALTERNATIVE FACTORY PAIRS


,Current Factory,Nearest Factory,Transactions
0,Lot's O' Nuts,Wicked Choccy's,1912
1,Wicked Choccy's,Lot's O' Nuts,1330
2,Lot's O' Nuts,Secret Factory,1020
3,Lot's O' Nuts,The Other Factory,879
4,Wicked Choccy's,Secret Factory,774
5,Wicked Choccy's,The Other Factory,609
6,Lot's O' Nuts,Sugar Shack,92
7,Secret Factory,Wicked Choccy's,78
8,Wicked Choccy's,Sugar Shack,64
9,Secret Factory,Lot's O' Nuts,62


In [5]:

# DAY 7 - FACTORY-TO-FACTORY OPPORTUNITY SUMMARY

factory_opportunities = (
    df[df["Current Factory"] != df["Nearest Factory"]]
    .groupby(["Current Factory", "Nearest Factory"])
    .agg(
        Transactions=("Row ID", "count"),
        Products_Involved=("Product Name", "nunique"),
        Average_Current_Distance=(
            "Current Factory Distance (km)", "mean"
        ),
        Average_Alternative_Distance=(
            "Nearest Factory Distance (km)", "mean"
        ),
        Average_Distance_Reduction=(
            "Potential Distance Reduction (km)", "mean"
        ),
        Total_Distance_Reduction=(
            "Potential Distance Reduction (km)", "sum"
        )
    )
    .reset_index()
    .sort_values(
        by="Total_Distance_Reduction",
        ascending=False
    )
)

print("FACTORY-TO-FACTORY OPPORTUNITY SUMMARY")
print("=" * 65)

display(factory_opportunities.round(2))

print("\nTOP 5 FACTORY PAIRS BY TOTAL POTENTIAL DISTANCE REDUCTION")

display(factory_opportunities.head(5).round(2))


FACTORY-TO-FACTORY OPPORTUNITY SUMMARY


,Current Factory,Nearest Factory,Transactions,Products_Involved,Average_Current_Distance,Average_Alternative_Distance,Average_Distance_Reduction,Total_Distance_Reduction
3,Lot's O' Nuts,Wicked Choccy's,1912,3,3271.17,935.77,2335.40,4465281.45
15,Wicked Choccy's,Lot's O' Nuts,1330,2,3449.08,935.37,2513.71,3343231.66
0,Lot's O' Nuts,Secret Factory,1020,3,2538.57,531.17,2007.40,2047544.39
2,Lot's O' Nuts,The Other Factory,879,3,1698.45,644.91,1053.54,926060.84
16,Wicked Choccy's,Secret Factory,774,2,1207.37,540.71,666.65,515988.88
18,Wicked Choccy's,The Other Factory,609,2,1267.88,633.27,634.60,386473.73
17,Wicked Choccy's,Sugar Shack,64,2,2618.71,792.38,1826.33,116885.35
1,Lot's O' Nuts,Sugar Shack,92,3,1987.81,989.63,998.18,91832.76
4,Secret Factory,Lot's O' Nuts,62,3,2342.66,946.59,1396.07,86556.40
12,The Other Factory,Lot's O' Nuts,33,2,2611.23,962.56,1648.67,54406.13



TOP 5 FACTORY PAIRS BY TOTAL POTENTIAL DISTANCE REDUCTION


,Current Factory,Nearest Factory,Transactions,Products_Involved,Average_Current_Distance,Average_Alternative_Distance,Average_Distance_Reduction,Total_Distance_Reduction
3,Lot's O' Nuts,Wicked Choccy's,1912,3,3271.17,935.77,2335.40,4465281.45
15,Wicked Choccy's,Lot's O' Nuts,1330,2,3449.08,935.37,2513.71,3343231.66
0,Lot's O' Nuts,Secret Factory,1020,3,2538.57,531.17,2007.40,2047544.39
2,Lot's O' Nuts,The Other Factory,879,3,1698.45,644.91,1053.54,926060.84
16,Wicked Choccy's,Secret Factory,774,2,1207.37,540.71,666.65,515988.88


In [6]:

# DAY 7 - OPTIMIZATION INPUT AVAILABILITY AUDIT

available_inputs = {
    "Product information": "Product Name",
    "Current factory assignment": "Current Factory",
    "Customer location": "Customer Location ID",
    "Current factory distance": "Current Factory Distance (km)",
    "Nearest factory distance": "Nearest Factory Distance (km)",
    "Sales information": "Sales",
    "Gross profit information": "Gross Profit",
    "Production quantity": "Units"
}

print("AVAILABLE OPTIMIZATION INPUTS")
print("=" * 55)

for description, column in available_inputs.items():
    print(
        description + ":",
        "Available" if column in df.columns else "Missing"
    )

print("\nADDITIONAL BUSINESS CONSTRAINTS")
print("=" * 55)

missing_inputs = [
    "Factory production capacity",
    "Product manufacturing compatibility by factory",
    "Factory-wise manufacturing cost",
    "Transportation cost per kilometre",
    "Factory operating limits"
]

for item in missing_inputs:
    print(item + ": Not available in current dataset")


AVAILABLE OPTIMIZATION INPUTS
Product information: Available
Current factory assignment: Available
Customer location: Available
Current factory distance: Available
Nearest factory distance: Available
Sales information: Available
Gross profit information: Available
Production quantity: Available

ADDITIONAL BUSINESS CONSTRAINTS
Factory production capacity: Not available in current dataset
Product manufacturing compatibility by factory: Not available in current dataset
Factory-wise manufacturing cost: Not available in current dataset
Transportation cost per kilometre: Not available in current dataset
Factory operating limits: Not available in current dataset


In [7]:

# DAY 7 - PRELIMINARY FACTORY REALLOCATION CANDIDATES

reallocation_candidates = (
    df[df["Current Factory"] != df["Nearest Factory"]]
    .groupby(
        ["Product Name", "Current Factory", "Nearest Factory"]
    )
    .agg(
        Transactions=("Row ID", "count"),
        Average_Current_Distance=(
            "Current Factory Distance (km)", "mean"
        ),
        Average_Alternative_Distance=(
            "Nearest Factory Distance (km)", "mean"
        ),
        Average_Distance_Reduction=(
            "Potential Distance Reduction (km)", "mean"
        ),
        Total_Distance_Reduction=(
            "Potential Distance Reduction (km)", "sum"
        ),
        Total_Sales=("Sales", "sum"),
        Total_Gross_Profit=("Gross Profit", "sum"),
        Total_Units=("Units", "sum")
    )
    .reset_index()
    .sort_values(
        by="Total_Distance_Reduction",
        ascending=False
    )
)

reallocation_candidates["Validation Status"] = (
    "Requires compatibility and capacity validation"
)

print("PRELIMINARY REALLOCATION CANDIDATES")
print("=" * 65)

display(reallocation_candidates.head(15).round(2))

print("\nTOTAL CANDIDATE COMBINATIONS:",
      len(reallocation_candidates))

print("TOTAL TRANSACTIONS WITH A CLOSER ALTERNATIVE:",
      reallocation_candidates["Transactions"].sum())


PRELIMINARY REALLOCATION CANDIDATES


,Product Name,Current Factory,Nearest Factory,Transactions,Average_Current_Distance,Average_Alternative_Distance,Average_Distance_Reduction,Total_Distance_Reduction,Total_Sales,Total_Gross_Profit,Total_Units,Validation Status
38,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,Lot's O' Nuts,667,3458.36,934.00,2524.35,1683744.04,9596.25,6269.55,2559,Requires compatibility and capacity validation
30,Wonka Bar - Milk Chocolate,Wicked Choccy's,Lot's O' Nuts,663,3439.74,936.75,2503.00,1659487.62,8700.25,5648.47,2677,Requires compatibility and capacity validation
45,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts,Wicked Choccy's,707,3287.82,953.19,2334.63,1650583.90,9579.60,6652.50,2661,Requires compatibility and capacity validation
29,Wonka Bar - Fudge Mallows,Lot's O' Nuts,Wicked Choccy's,620,3274.11,937.46,2336.66,1448728.15,8362.80,5575.20,2323,Requires compatibility and capacity validation
37,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts,Wicked Choccy's,585,3247.92,912.93,2334.99,1365969.41,7646.59,5455.59,2191,Requires compatibility and capacity validation
34,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts,Secret Factory,347,2504.05,503.27,2000.79,694273.35,4575.39,3264.39,1311,Requires compatibility and capacity validation
42,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts,Secret Factory,346,2546.85,543.99,2002.86,692988.90,4690.80,3257.50,1303,Requires compatibility and capacity validation
26,Wonka Bar - Fudge Mallows,Lot's O' Nuts,Secret Factory,327,2566.43,547.22,2019.21,660282.15,4431.60,2954.40,1231,Requires compatibility and capacity validation
44,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts,The Other Factory,316,1687.78,655.51,1032.27,326195.88,4406.40,3060.00,1224,Requires compatibility and capacity validation
28,Wonka Bar - Fudge Mallows,Lot's O' Nuts,The Other Factory,281,1717.58,645.86,1071.73,301155.46,3819.60,2546.40,1061,Requires compatibility and capacity validation



TOTAL CANDIDATE COMBINATIONS: 49
TOTAL TRANSACTIONS WITH A CLOSER ALTERNATIVE: 6967


In [8]:

# DAY 7 - SAVE PRELIMINARY REALLOCATION DATASETS

candidate_file = "/content/preliminary_reallocation_candidates.csv"
factory_file = "/content/factory_opportunity_summary.csv"

reallocation_candidates.to_csv(candidate_file, index=False)
factory_opportunities.to_csv(factory_file, index=False)

print("FILES SAVED SUCCESSFULLY")
print("=" * 50)

print("1. preliminary_reallocation_candidates.csv")
print("   Rows:", len(reallocation_candidates))

print("\n2. factory_opportunity_summary.csv")
print("   Rows:", len(factory_opportunities))

print("\nTotal candidate combinations:",
      len(reallocation_candidates))

print("Transactions with closer alternatives:",
      reallocation_candidates["Transactions"].sum())


FILES SAVED SUCCESSFULLY
1. preliminary_reallocation_candidates.csv
   Rows: 49

2. factory_opportunity_summary.csv
   Rows: 19

Total candidate combinations: 49
Transactions with closer alternatives: 6967


# Day 7: Model Development, Evaluation and Preliminary Reallocation Analysis

## 1. Objective
The objective of Day 7 was to examine whether machine learning was suitable for the factory reallocation problem and identify potential product-to-factory reassignment opportunities using geographical distance and business performance data.

## 2. Model Suitability Assessment
The available dataset contains product information, current factory assignments, customer locations, factory distances, sales, gross profit and production quantities.

However, it does not contain verified target variables for predicting shipping costs, delivery times or optimal factory assignments.

Additionally, reliable shipping lead-time information could not be established during earlier data validation.

Therefore, a supervised machine learning model was not forced into the project. The problem is better approached initially as a geographical analysis and optimization problem.

## 3. Product-wise Opportunity Analysis
Products were grouped by their current factory assignments to evaluate transaction counts, average distances, potential distance reductions, sales and gross profit.

The analysis identified substantial geographical improvement opportunities, particularly among the high-volume Wonka Bar products.

## 4. Factory-to-Factory Opportunity Analysis
A factory-level summary was generated by grouping transactions according to their current factory and nearest alternative factory.

The top geographical opportunities were:

| Current Factory | Alternative Factory | Transactions | Average Distance Reduction |
|---|---|---:|---:|
| Lot's O' Nuts | Wicked Choccy's | 1,912 | 2,335.40 km |
| Wicked Choccy's | Lot's O' Nuts | 1,330 | 2,513.71 km |
| Lot's O' Nuts | Secret Factory | 1,020 | 2,007.40 km |
| Lot's O' Nuts | The Other Factory | 879 | 1,053.54 km |
| Wicked Choccy's | Secret Factory | 774 | 666.65 km |

These values represent potential geographical distance reductions, not confirmed transportation savings.

## 5. Optimization Input Availability

### Available inputs
- Product information
- Current factory assignments
- Customer locations
- Current factory distances
- Nearest factory distances
- Sales information
- Gross profit information
- Production quantities

### Missing business constraints
- Factory production capacity
- Product manufacturing compatibility by factory
- Factory-wise manufacturing costs
- Transportation cost per kilometre
- Factory operating limits

These missing constraints must be considered before making final factory allocation decisions.

## 6. Preliminary Factory Reallocation Candidates
A preliminary recommendation dataset was created by identifying transactions where the nearest factory differed from the currently assigned factory.

The analysis generated:

- 49 product–factory candidate combinations.
- 6,967 transactions with a closer alternative factory.
- 3,227 transactions already assigned to their nearest factory.

The five highest-priority geographical candidates involved the following products:

1. Wonka Bar – Triple Dazzle Caramel
2. Wonka Bar – Milk Chocolate
3. Wonka Bar – Scrumdiddlyumptious
4. Wonka Bar – Fudge Mallows
5. Wonka Bar – Nutty Crunch Surprise

These candidates require manufacturing compatibility and capacity validation before they can be considered feasible recommendations.

## 7. Files Generated

The following output files were generated successfully:

1. `preliminary_reallocation_candidates.csv` – Contains 49 preliminary product–factory combinations.
2. `factory_opportunity_summary.csv` – Contains 19 factory-to-factory opportunity pairs.

## 8. Limitations
- Factory distances are calculated using the Haversine formula and represent straight-line geographical distances.
- Actual road distances, transportation times and transportation costs are not available.
- The nearest factory is not necessarily capable of manufacturing every product.
- Production capacity and factory operating constraints are unavailable.
- Potential distance reductions should not be interpreted as confirmed operational or financial savings.

## 9. Conclusion
Day 7 established that the available dataset is suitable for geographical factory comparison and preliminary reallocation analysis.

Rather than forcing an unsuitable supervised machine learning model, the project will proceed towards a transparent optimization and recommendation framework.

The next stage will compare existing factory allocations with nearest-factory scenarios and gradually incorporate business constraints wherever reliable information becomes available.

This provides the foundation for developing the Factory Reallocation & Shipping Optimization Recommendation System.